In [1]:
bt_enabled = True  # Set to False to stop

from pynq import Overlay, MMIO
from time import sleep, time

In [2]:
ol = Overlay("car.bit", ignore_version=True)
print("✔ Overlay 'car.bit' loaded successfully.\n")

✔ Overlay 'car.bit' loaded successfully.



In [3]:
# Access the AXI GPIO block (used for GPS enable/reset)
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000, debug=False)

In [4]:
# AXI UARTLite base address
uart = MMIO(ol.Bluetooth.mmio.base_addr, 0x1000, debug=False)

print("✔ AXI UARTLite MMIO mapped")

✔ AXI UARTLite MMIO mapped


In [5]:
# Reset TX and RX FIFOs
uart.write(0x0C, 0x03)
sleep(0.1)

In [6]:
def uart_tx_byte(byte):
    # Wait until TX FIFO is not full
    while uart.read(0x08) & 0x08:
        pass
    uart.write(0x04, byte)

def uart_rx_byte():
    # Check if RX FIFO has data
    if uart.read(0x08) & 0x01:
        return uart.read(0x00) & 0xFF
    return None

def uart_send_string(s):
    for c in s:
        uart_tx_byte(ord(c))
        
def uart_read_string(timeout_s=0.5, idle_gap_s=0.05):
    out = []
    start = time()
    last_rx = None

    while (time() - start) < timeout_s:
        b = uart_rx_byte()
        if b is not None:
            out.append(chr(b))
            last_rx = time()
        else:
            # If we've started receiving and then go idle long enough, stop
            if last_rx is not None and (time() - last_rx) > idle_gap_s:
                break
            sleep(0.005)  # tiny pause so we don't spin at 100% CPU

    return "".join(out)

In [7]:
#Don't need to run again
uart_send_string("AT")
sleep(0.2)
response = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    response += chr(b)

print("Bluetooth response:", response)

Bluetooth response: OK


In [8]:
uart_send_string("AT+BATT?")   # \r\n often helps
resp = uart_read_string(timeout_s=0.5, idle_gap_s=0.05)
print("Bluetooth response:", repr(resp))

Bluetooth response: 'OK+Get:100'


In [20]:
uart_send_string("AT+BAUD?")   # \r\n often helps
resp = uart_read_string(timeout_s=0.5, idle_gap_s=0.05)
print("Bluetooth response:", repr(resp))

Bluetooth response: 'OK+Get:0'


In [9]:
#Don't need to run again
uart_send_string("AT+ROLE0")
sleep(0.2)

In [10]:
uart_send_string("AT+ROLE?")
sleep(0.2)

resp = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    resp += chr(b)

print("ROLE response:", resp)

ROLE response: OKOK+Set:0OK+Get


In [11]:
uart_send_string("AT+ADDR?")
sleep(0.2)

resp = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    resp += chr(b)

print("Address response:", resp)

Address response: OK+ADDR:685E1C26
